## Install ANTLR

In [11]:
# --- 1. Install Python Runtime ---
# Install the runtime version that matches the ANTLR tool version
!pip install antlr4-python3-runtime==4.13.1

# --- 2. Download ANTLR Tool (Java JAR) ---
# We'll download version 4.13.1, but any recent version should work
ANTLR_JAR = "antlr-4.13.1-complete.jar"
!curl -O https://www.antlr.org/download/{ANTLR_JAR}

# --- 3. Create a Bash Alias (for easy use of the ANTLR tool) ---
# We'll save the alias to a temporary file, then source it
ALIAS_COMMAND = f"alias antlr4='java -jar {ANTLR_JAR}'"
!echo "{ALIAS_COMMAND}" > ~/.bashrc
!source ~/.bashrc

print("Setup complete! ANTLR tool is ready to use.")

1186.36s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


1192.08s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2089k  100 2089k    0     0  2560k      0 --:--:-- --:--:-- --:--:-- 2560k


1199.40s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
1204.77s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Setup complete! ANTLR tool is ready to use.


In [12]:
!ls -F

1214.39s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Doggo.g4                   scratch.ipynb
antlr-4.13.1-complete.jar


In [13]:
%%writefile Doggo.g4
grammar Doggo;

// Parser Rules (Start with lowercase: ANTLR needs these to generate the Parser)
start: expr EOF;

expr
    : expr MUL expr # MulDiv
    | expr DIV expr # MulDiv
    | expr ADD expr # AddSub
    | expr SUB expr # AddSub
    | atom # AtomicExpression // Renamed label to avoid conflict
    ;

atom
    : INT # Number
    | '(' expr ')' # Parentheses
    ;

// Lexer Rules (Start with uppercase)
ADD : '+' ;
SUB : '-' ;
MUL : '*' ;
DIV : '/' ;

INT : [0-9]+ ;
WS  : [ \t\r\n]+ -> skip ;

Overwriting Doggo.g4


In [14]:
# Execute the ANTLR tool to generate the Python source files
# -Dlanguage=Python3: Specifies the target language
# -visitor: Generates the DoggoLexerVisitor base class
# The generated files will appear in the Colab file explorer.

!java -jar antlr-4.13.1-complete.jar -Dlanguage=Python3 -visitor Doggo.g4

print("ANTLR Python code generated successfully.")

1232.30s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


ANTLR Python code generated successfully.


In [15]:
!ls

1244.08s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Doggo.g4                  DoggoLexer.py             DoggoVisitor.py
Doggo.interp              DoggoLexer.tokens         antlr-4.13.1-complete.jar
Doggo.tokens              DoggoListener.py          scratch.ipynb
DoggoLexer.interp         DoggoParser.py


In [16]:
from antlr4 import *
from antlr4.error.ErrorListener import ErrorListener # Import the ErrorListener class
from DoggoLexer import DoggoLexer
from DoggoParser import DoggoParser
from DoggoVisitor import DoggoVisitor # This is the base class, we'll use our custom interpreter below
import sys

# Custom Error Listener to suppress default error messages
class SuppressErrorListener(ErrorListener): # Inherit from ErrorListener
    def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
        pass

    def reportAmbiguity(self, recognizer, dfa, startIndex, stopIndex, exact, ambigAlts, configs):
        pass

    def reportAttemptingFullContext(self, recognizer, dfa, startIndex, stopIndex, conflictingAlts, configs):
        pass

    def reportContextSensitivity(self, recognizer, dfa, startIndex, stopIndex, prediction, configs):
        pass

# Override the Visitor methods to perform the calculations
class DoggoInterpreter(DoggoVisitor):

    # Visit a parse tree produced by DoggoParser#start.
    def visitStart(self, ctx:DoggoParser.StartContext):
        return self.visit(ctx.expr())

    # Visit a parse tree produced by DoggoParser#MulDiv.
    def visitMulDiv(self, ctx:DoggoParser.MulDivContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        # Check the token type for multiplication or division
        op_token_type = ctx.getChild(1).getSymbol().type
        if op_token_type == DoggoLexer.MUL:
            return left * right
        elif op_token_type == DoggoLexer.DIV:
            # Handle division by zero if necessary
            if right == 0:
                raise ValueError("Division by zero")
            return left / right

    # Visit a parse tree produced by DoggoParser#AddSub.
    def visitAddSub(self, ctx:DoggoParser.AddSubContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        # Check the token type for addition or subtraction
        op_token_type = ctx.getChild(1).getSymbol().type
        if op_token_type == DoggoLexer.ADD:
            return left + right
        elif op_token_type == DoggoLexer.SUB:
            return left - right


    # Visit a parse tree produced by DoggoParser#Number.
    def visitNumber(self, ctx:DoggoParser.NumberContext):
        return float(ctx.INT().getText())

    # Visit a parse tree produced by DoggoParser#Parentheses.
    def visitParentheses(self, ctx:DoggoParser.ParenthesesContext):
        return self.visit(ctx.expr())


def run_interpreter(input_string):
    # 1. Create a stream of characters from the input
    input_stream = InputStream(input_string)

    # 2. Create the Lexer (tokenizer)
    lexer = DoggoLexer(input_stream)
    # Remove default error listeners and add our custom one
    lexer.removeErrorListeners()
    lexer.addErrorListener(SuppressErrorListener())


    # 3. Create a stream of tokens
    token_stream = CommonTokenStream(lexer)

    # 4. Create the Parser
    parser = DoggoParser(token_stream)
    # Remove default error listeners and add our custom one
    parser.removeErrorListeners()
    parser.addErrorListener(SuppressErrorListener())


    # 5. Get the root of the Parse Tree by calling the starting rule
    tree = parser.start()

    # Print the parse tree
    print("--- Parse Tree ---")
    print(tree.toStringTree(recog=parser))
    print("------------------")

    # 6. Create our custom Interpreter (Visitor) and start the traversal (interpretation)
    interpreter = DoggoInterpreter() # Use our custom interpreter
    result = interpreter.visit(tree)


    return result

# --- Test the Interpreter ---
input_expression = "10 + 5 * (8 - 4) / 2" # Should evaluate to 20.0

print(f"--- Running Simple Expression Interpreter ---")
print(f"Input: {input_expression}")

try:
    output = run_interpreter(input_expression)
    print(f"Result: {output}")

    # Another test
    input_expression_2 = "(100 - 10) / 9 + 1" # Should evaluate to 11.0
    output_2 = run_interpreter(input_expression_2)
    print(f"\nInput: {input_expression_2}")
    print(f"Result: {output_2}")

except Exception as e:
    print(f"Error during interpretation: {e}")

--- Running Simple Expression Interpreter ---
Input: 10 + 5 * (8 - 4) / 2
--- Parse Tree ---
(start (expr (expr (atom 10)) + (expr (expr (expr (atom 5)) * (expr (atom ( (expr (expr (atom 8)) - (expr (atom 4))) )))) / (expr (atom 2)))) <EOF>)
------------------
Result: 20.0
--- Parse Tree ---
(start (expr (expr (expr (atom ( (expr (expr (atom 100)) - (expr (atom 10))) ))) / (expr (atom 9))) + (expr (atom 1))) <EOF>)
------------------

Input: (100 - 10) / 9 + 1
Result: 11.0
